<a href="https://colab.research.google.com/github/VakitiSriVarsha/Gen-AI/blob/main/Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q requests beautifulsoup4 sentence-transformers

In [ ]:
agent_memory = []

In [ ]:
import requests
from bs4 import BeautifulSoup

def fetch_web_content(url):
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    for tag in soup(["script", "style"]):
        tag.decompose()

    paragraphs = soup.find_all("p")
    text = " ".join(para.get_text().strip() for para in paragraphs)

    return text

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

def summarize_text(text, max_sentences=3):

    sentences = [s.strip() for s in text.split(".") if len(s.strip()) > 30]

    if len(sentences) == 0:
        return "No sufficient content could be extracted to summarize."

    embeddings = model.encode(sentences)

    if len(embeddings) == 0:
        return "Embedding failed due to empty content."

    doc_embedding = np.mean(embeddings, axis=0)

    scores = embeddings @ doc_embedding
    top_indices = np.argsort(scores)[-max_sentences:]

    summary = ". ".join(sentences[i] for i in sorted(top_indices))
    return summary

In [ ]:

def agent(query):
    agent_memory.append(f"User Query: {query}")

    agent_memory.append("Reasoning: Need to fetch data and summarize it")

    url = "https://en.wikipedia.org/wiki/Artificial_intelligence"
    agent_memory.append(f"Action: Fetching from {url}")

    fetched_text = fetch_web_content(url)
    agent_memory.append("Observation: Data fetched")

    summary = summarize_text(fetched_text)
    agent_memory.append("Action: Summary generated")

    return summary


In [ ]:
result = agent("Explain AI agents")

print("✅ Agent Response:\n")
print(result)

print("\n🧠 Agent Memory:\n")
for step in agent_memory:
    print("-", step)